# The SVD and PCA: compressing data and finding patterns

*SIAM Data-Driven Modeling Group, 11/11/26. Reading: Brunton & Kutz, Chapter 1.*

Big data sets are often secretly simple: millions of numbers described by a handful of patterns. The **singular value decomposition (SVD)** finds those patterns. Today we use it to compress a photo, find "eigenfaces", and describe hot metal rods with a few numbers.

We'll go through this notebook together on the screen. Feel free to run the cells along with us (in a **copy** of the notebook; see the README in `10_21_26`). The **toolbox** at the end of Part 4 collects every command you need to try this on your own data.

1. Data as vectors (and why we need compression)
2. The goal of compression
3. The SVD and PCA
4. The mathematics
5. Applications: compressing a photo, eigenfaces, and hot rods

## Setup

Run this cell first (and again after restarting). It imports the libraries and defines small helpers for loading and showing images.

In [ ]:
import numpy as np                 # arrays and linear algebra
import matplotlib.pyplot as plt    # plotting
import scipy.io                    # reads MATLAB .mat files (for the faces data)

DATA_DIR = "../DATA"               # the DATA folder sits one level above this notebook's folder

# Plot colors (a colorblind-friendly set)
BLUE, ORANGE, GREEN, YELLOW, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8c8c8c"


def load_gray(path):
    """Load an image file as a 2D grayscale array of decimal numbers (0 = black, 255 = white)."""
    A = plt.imread(path)
    if A.dtype != np.uint8:        # some formats (like PNG) load as numbers from 0 to 1
        A = 255 * A
    if A.ndim == 3:                # a color image: combine red, green and blue (Part 1 explains how)
        A = A[:, :, :3] @ np.array([0.2989, 0.5870, 0.1140])
    return np.asarray(A, dtype=float)


def show_image(img, title=None, ax=None, vmin=None, vmax=None):
    """Show a 2D array as a grayscale picture."""
    if ax is None:
        fig, ax = plt.subplots()
    ax.imshow(img, cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")


def show_images(images, titles=None, ncols=4, size=3):
    """Show a list of 2D arrays side by side, in a grid with ncols columns."""
    nrows = int(np.ceil(len(images) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, 1.15 * size * nrows), squeeze=False)
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            show_image(images[i], None if titles is None else titles[i], ax=ax)
        else:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

# Part 1: Data as vectors (and why we need compression)

## 1.1 A picture is a matrix of numbers

Meet Mordecai the snow dog, the book's running example. `plt.imread` loads the photo as a NumPy **array**: a grid of numbers.

In [ ]:
A = plt.imread(DATA_DIR + "/dog.jpg")

print("shape:", A.shape)    # (rows, columns, colors)
print("type: ", A.dtype)    # uint8 = whole numbers from 0 to 255

The shape `(2000, 1500, 3)` means 2000 rows and 1500 columns of pixels, with 3 numbers per pixel: how much **red, green and blue** light it has. Each is a whole number from 0 (none) to 255 (full brightness).

To keep things simple we'll work in grayscale, with one brightness number per pixel. We get it from a weighted average of the three colors. The eye is most sensitive to green, so green gets the biggest weight. (These are the weights of MATLAB's `rgb2gray`, which the book uses.)

The `@` symbol is **matrix multiplication**. Here it multiplies the 3 color values of every pixel by the 3 weights and adds them up.

In [ ]:
weights = np.array([0.2989, 0.5870, 0.1140])    # red, green, blue
X = A @ weights                                  # one brightness value per pixel

print("shape:", X.shape)
print("type: ", X.dtype)                         # float64 = ordinary decimal numbers
show_image(X, "Mordecai, as a 2000 x 1500 matrix")

`X` is now an ordinary **matrix** with 2000 rows and 1500 columns. Let's zoom in on Mordecai's eye and look at the actual numbers. In NumPy:
- `X[i, j]` is the entry in row `i`, column `j`. Python counts from 0, so the top-left entry is `X[0, 0]`.
- `X[a:b, c:d]` is a **slice**: rows `a` up to (but not including) `b`, and columns `c` up to (but not including) `d`.

In [ ]:
eye = X[560:720, 440:600]          # a 160 x 160 block around the eye
patch = X[620:630, 500:510]        # a 10 x 10 block inside it

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
show_image(X, "the whole matrix", ax=axes[0])
axes[0].add_patch(plt.Rectangle((440, 560), 160, 160, fill=False, color=ORANGE, lw=2))
show_image(eye, "rows 560-719, columns 440-599", ax=axes[1])
axes[1].add_patch(plt.Rectangle((500 - 440 - 0.5, 620 - 560 - 0.5), 10, 10, fill=False, color=ORANGE, lw=2))
show_image(patch, "rows 620-629, columns 500-509", ax=axes[2], vmin=0, vmax=255)
for i in range(10):
    for j in range(10):
        value = patch[i, j]
        axes[2].text(j, i, f"{value:.0f}", ha="center", va="center", fontsize=8,
                     color="black" if value > 128 else "white")
plt.show()

## 1.2 A picture is also a single vector

We can also **unroll** the matrix into one long list of numbers. `reshape(-1)` lays the rows end to end (the `-1` means "work out the length for me").

In [ ]:
x = X.reshape(-1)           # unroll the matrix, one row after another

print("x has shape", x.shape)
print("its first five entries:", x[:5])

So the whole photo is **one vector with 3,000,000 entries**: a single point in $\mathbb{R}^{3{,}000{,}000}$. Nothing is lost, since `x.reshape(2000, 1500)` turns it back into the picture.

Both views will come back later:
- **The matrix view** (compressing the dog): we treat the photo as a matrix.
- **The vector view** (eigenfaces): in a collection of photos of faces, each photo becomes one long vector.

### Most of $\mathbb{R}^{3{,}000{,}000}$ looks like static

Here's a thought experiment. Pick a point in $\mathbb{R}^{3{,}000{,}000}$ **at random** (every pixel gets an independent, random brightness) and look at it as a picture:

In [ ]:
rng = np.random.default_rng(0)                  # a random number generator (seed 0, so we all get the same numbers)
noise = rng.uniform(0, 255, size=X.shape)       # a random point in the same space as the dog

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
show_image(X, "a photo", ax=axes[0])
show_image(noise, "a random point in the same space", ax=axes[1])
plt.show()

You could draw random points all day and never get anything that looks like a photo. Natural images fill only a tiny, highly structured corner of this enormous space: neighboring pixels tend to be similar, there are smooth regions and sharp edges, and so on. **That structure is what makes compression possible.** If a photo could be *any* point in $\mathbb{R}^{3{,}000{,}000}$, there would be no way around storing all 3 million numbers.

And storing them all takes a lot of space:

In [ ]:
print(f"The dog matrix holds {X.size:,} numbers and takes up {X.nbytes / 1e6:.0f} MB of memory.")

(The file `dog.jpg` is under 1 MB, and it's in color! That's because JPEG is itself a compression method. It uses a cousin of the Fourier transform, which comes up later in the book.)

## 1.3 High-dimensional data is everywhere

- **Movies:** one 4K video frame has 3840 × 2160 pixels × 3 colors ≈ 25 million numbers, and there are 24 to 60 frames every second.
- **Scientific simulations:** a weather or fluid simulation on a 3D grid of 1000 × 1000 × 100 points, storing 5 quantities at each point (3 velocity components, pressure, temperature), has 500 million numbers *per time step*.
- **Medicine and biology:** 3D MRI scans, or the activity of thousands of genes for each patient (one of the book's examples has 4000 genes per patient).
- **Neuroscience:** recordings from thousands of neurons at once, thousands of times per second.

In each case a single measurement is a vector in $\mathbb{R}^n$, with $n$ in the millions or billions. And just like photos, the data usually has structure: it lies close to a much **lower-dimensional subspace**, built from a few dominant patterns. The rest of today is about finding those patterns.

# Part 2: The goal of compression

Here are 200 points in $\mathbb{R}^2$, stored as the **columns** of a $2 \times 200$ matrix $X$. We'll use this convention all day: **each column is one data point**.

The points lie close to a line through the origin. If $u$ is a unit vector along that line, we can replace each point $x_i$ by **one number**, its coordinate $c_i = u^Tx_i$, and rebuild it as $x_i \approx c_i\,u$: that's $200 + 2$ numbers instead of $400$. The price is the total squared error (the orange segments below)
$$E(u) = \sum_{i=1}^N \big\| x_i - (u^Tx_i)\,u \big\|^2 ,$$
and the **best line** is the one that makes $E$ smallest.

In [ ]:
def make_line_data(N=200, seed=1):
    """N points in R^2 scattered around a line through the origin (one point per column)."""
    rng = np.random.default_rng(seed)
    direction = np.array([np.cos(np.pi / 6), np.sin(np.pi / 6)])   # a unit vector, 30 degrees up
    t = rng.normal(0, 2, size=N)                                   # how far along the line each point is
    noise = rng.normal(0, 0.4, size=(2, N))                        # random scatter off the line
    return np.outer(direction, t) + noise    # np.outer(a, b) = a b^T, so column i is t[i] * direction


def fit_line(X, u):
    """Replace every column of X by its closest point on the line along the unit vector u.
    Returns the approximation and the total squared error E(u)."""
    X_approx = np.outer(u, u @ X)                    # column i is (u^T x_i) u
    return X_approx, np.sum((X - X_approx) ** 2)


X = make_line_data()
N = X.shape[1]
U, S, Vt = np.linalg.svd(X, full_matrices=False)     # the SVD (Part 3). u_1 is U[:, 0], u_2 is U[:, 1]
bad_u = np.array([np.cos(np.radians(120)), np.sin(np.radians(120))])    # a line at 120 degrees

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharex=True, sharey=True)
axes[0].set_title(r"the data: 200 points in $\mathbb{R}^2$")
for ax, u, name in [(axes[1], bad_u, "a bad line"), (axes[2], U[:, 0], "the best line, from the SVD")]:
    X_approx, error = fit_line(X, u)
    ax.plot([-8 * u[0], 8 * u[0]], [-8 * u[1], 8 * u[1]], color=BLUE, lw=3, alpha=0.35)   # the line
    ax.plot([X[0], X_approx[0]], [X[1], X_approx[1]], color=ORANGE, lw=0.8)               # the errors
    ax.set_title(f"{name}: E = {error:.0f}")
for k, color in enumerate([BLUE, GREEN]):                     # u_1 and u_2, with length sigma_k / sqrt(N)
    arrow = S[k] / np.sqrt(N) * U[:, k]
    axes[2].arrow(0, 0, arrow[0], arrow[1], width=0.12, color=color, length_includes_head=True,
                  label=f"$u_{k + 1}$", zorder=4)
axes[2].legend(loc="lower right")
for ax in axes:
    ax.scatter(X[0], X[1], s=12, color=GRAY, alpha=0.7, zorder=3)   # X[0] = first coordinates, X[1] = second ones
    ax.set_aspect("equal"); ax.set_xlim(-5.5, 5.5); ax.set_ylim(-4, 4)
plt.show()

In 2D we could find the best line by trying every angle. But a photo's columns live in $\mathbb{R}^{2000}$, and we may want the best 100-dimensional subspace rather than the best line. **How do we find the best subspace directly?**

**Preview of the answer: the SVD.** Its first left singular vector $u_1$ points along the best line (right panel), and $u_2$ points across it. The arrows have length $\sigma_k / \sqrt{N}$, where the **singular values** $\sigma_1 \geq \sigma_2$ measure how spread out the data is in each direction.

# Part 3: The SVD and PCA

## 3.1 The SVD

Every real $n \times m$ matrix $X$ can be factored as
$$X = U \Sigma V^T ,$$
- $U$ ($n\times n$) is orthogonal. Its columns $u_1, u_2, \dots$, the **left singular vectors**, form an orthonormal basis of $\mathbb{R}^n$: the space the data points live in.
- $\Sigma$ ($n \times m$) is diagonal, with the **singular values** $\sigma_1 \geq \sigma_2 \geq \dots \geq 0$ on its diagonal.
- $V$ ($m \times m$) is orthogonal. Its columns are the **right singular vectors**.

We almost always use the **economy SVD**, which keeps only the first $\min(n, m)$ columns of $U$ and $V$ (the others only ever get multiplied by zeros).

In NumPy, `U, S, Vt = np.linalg.svd(X, full_matrices=False)`. Watch out: `S` is a 1D array of the $\sigma_k$ (use `np.diag(S)` for the matrix $\Sigma$), and `Vt` is $V^T$, not $V$.

In [ ]:
U, S, Vt = np.linalg.svd(X, full_matrices=False)      # X = the 2 x 200 data from Part 2

print("U:", U.shape, "  S:", S.shape, "  Vt:", Vt.shape)
print("singular values:", S)
print("U Sigma V^T = X ?", np.allclose(U @ np.diag(S) @ Vt, X))     # equal up to rounding errors?

## 3.2 What the SVD tells us about data

Think of the columns $x_1, \dots, x_m$ of $X$ as data points in $\mathbb{R}^n$, and let $U_r$ hold the first $r$ columns of $U$.

- **Best subspaces.** $u_1$ spans the best line through the origin (the smallest total squared distance to the points). $u_1, u_2$ span the best plane, and in general $u_1, \dots, u_r$ span the best $r$-dimensional subspace.
- **Compression.** A point $x$ is stored as its $r$ coordinates $c = U_r^Tx$ in that subspace, and rebuilt as $U_rc = U_rU_r^Tx$ (its projection onto the subspace).
- **Singular values measure spread.** $\sigma_k^2 = \sum_i (u_k^Tx_i)^2$ is the total squared length of the data along $u_k$. The error of the best $r$-dimensional fit is whatever is left over:
$$\sum_i \big\| x_i - U_rU_r^Tx_i \big\|^2 = \sigma_{r+1}^2 + \sigma_{r+2}^2 + \cdots$$
So if the singular values drop off fast, a few directions describe the data well.
- **All the points at once.** Projecting every column of $X$ gives the **truncated SVD**
$$X_r = U_rU_r^TX = U_r\Sigma_rV_r^T ,$$
where $\Sigma_r$ is the top-left $r\times r$ block of $\Sigma$ and $V_r^T$ is the first $r$ rows of $V^T$. Storing $U_r$, $\Sigma_r$ and $V_r$ instead of $X$ is how the SVD compresses data.

For the points from Part 2 ($r = 1$), the error of the best line is exactly $\sigma_2^2$:

In [ ]:
r = 1
X_r = U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]          # every point, projected onto the line along u_1

print("error of the best line:", np.sum((X - X_r)**2))
print("sigma_2 squared:       ", S[1]**2)

## 3.3 PCA: the SVD of centered data

The SVD finds the best subspaces **through the origin**. If the data sits far from the origin, $u_1$ mostly points from the origin towards the data: it describes *where* the data is, not *how it varies*. **Principal component analysis (PCA)** fixes this by removing the mean first:
1. Compute the mean data point $\bar x = \frac{1}{N}\sum_i x_i$.
2. Subtract it from every column: $B = X - \bar x \mathbf{1}^T$.
3. Take the SVD $B/\sqrt{N} = U\Sigma V^T$.

Now the $u_k$, called the **principal directions**, describe how the data varies around its mean, and $\sigma_k$ is the **standard deviation** of the data along $u_k$.

Here's a cloud of 10,000 points centered at $(2, 1)$, with standard deviations 2 and 0.5 along axes rotated by 60°:

In [ ]:
rng = np.random.default_rng(4)
N = 10_000
theta = np.pi / 3
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])                  # rotation by 60 degrees
X = R @ np.diag([2, 0.5]) @ rng.normal(size=(2, N)) + np.array([[2], [1]])   # stretch, rotate, shift

# PCA
x_bar = X.mean(axis=1)                         # the mean data point, shape (2,)
B = X - x_bar[:, np.newaxis]                   # subtract it from every column ("broadcasting")
U, S, Vt = np.linalg.svd(B / np.sqrt(N), full_matrices=False)
print("PCA: mean =", x_bar, "  standard deviations =", S)

# Plain SVD, without centering (for comparison)
U_raw, S_raw, _ = np.linalg.svd(X / np.sqrt(N), full_matrices=False)
U_raw = U_raw * np.sign(U_raw[:, 0] @ x_bar)  # flip signs if needed, so u_1 points towards the data

fig, axes = plt.subplots(1, 2, figsize=(11, 6.5), sharex=True, sharey=True)
for ax, base, Uk, Sk, title in [(axes[0], [0, 0], U_raw, S_raw, "plain SVD (through the origin)"),
                                 (axes[1], x_bar, U, S, "PCA (through the mean)")]:
    ax.scatter(X[0], X[1], s=1, color=GRAY, alpha=0.3)
    for k, color in enumerate([BLUE, ORANGE]):
        a = Sk[k] * Uk[:, k]
        ax.arrow(base[0], base[1], a[0], a[1], width=0.08, color=color, length_includes_head=True,
                 label=rf"$\sigma_{k + 1} u_{k + 1}$", zorder=3)
    ax.set_title(title); ax.set_aspect("equal"); ax.legend(loc="lower right")
plt.show()

| | **SVD** | **PCA** |
|---|---|---|
| what it is | a factorization of any matrix | a recipe for data: subtract the mean, then take the SVD |
| best-fit subspaces pass through | the origin | the mean $\bar x$ |
| $\sigma_k$ measures | the size of the data along $u_k$ | the standard deviation of the data along $u_k$ |
| use it when | zero is meaningful (an image, a temperature field) | you care about variation around the average (faces, statistics) |

*Note:* scikit-learn and most statistics texts store data points as **rows**. Then everything is transposed, and the principal directions are the columns of $V$.

# Part 4: The mathematics

Three theorems behind Part 3. Matrices are real, and $\|A\|_F^2 = \sum_{i,j} A_{ij}^2$ is the squared **Frobenius norm**.

**Theorem 1 (the SVD exists).** Every real $n\times m$ matrix $X$ can be written as $X = U\Sigma V^T$, with $U, V$ orthogonal and $\Sigma$ diagonal with entries $\sigma_1 \geq \sigma_2 \geq \dots \geq 0$. Moreover
$$X^TX = V(\Sigma^T\Sigma)V^T, \qquad XX^T = U(\Sigma\Sigma^T)U^T ,$$
so the $u_k$ are eigenvectors of $XX^T$ with eigenvalues $\sigma_k^2$. (In PCA, $BB^T/N$ is the covariance matrix: the principal directions are its eigenvectors.) The rank of $X$ is the number of nonzero $\sigma_k$.

<details>
<summary><b>Proof</b></summary>

1. $X^TX$ is symmetric, with eigenvalues $\geq 0$ because $v^TX^TXv = \|Xv\|^2 \geq 0$. By the spectral theorem, $X^TX = V\Lambda V^T$ with $V$ orthogonal and $\lambda_1 \geq \dots \geq \lambda_m \geq 0$.
2. Let $\sigma_k = \sqrt{\lambda_k}$, and let $r$ be the number of positive $\lambda_k$. For $k \leq r$ define $u_k = Xv_k/\sigma_k$. These are orthonormal: $u_j^Tu_k = v_j^TX^TXv_k/(\sigma_j\sigma_k) = \lambda_k v_j^Tv_k/(\sigma_j\sigma_k) = \delta_{jk}$.
3. For $k > r$, $\|Xv_k\|^2 = \lambda_k = 0$, so $Xv_k = 0$.
4. Extend $u_1, \dots, u_r$ to an orthonormal basis $u_1, \dots, u_n$ of $\mathbb{R}^n$, and put $\sigma_1, \dots, \sigma_r$ on the diagonal of the $n \times m$ matrix $\Sigma$.
5. Then $XV = U\Sigma$ column by column, so $X = U\Sigma V^T$. $\blacksquare$

The formulas for $X^TX$ and $XX^T$ follow by multiplying out. Since $U$ and $V$ are invertible, $\operatorname{rank} X = \operatorname{rank}\Sigma = r$.

</details>

*Signs:* each pair $(u_k, v_k)$ is only unique up to a sign flip of both. So NumPy's vectors may point the opposite way from a textbook's (eigenfaces can come out as photo negatives).

**Theorem 2 (orthogonal projection).** Let $Q$ be $n \times r$ with orthonormal columns ($Q^TQ = I_r$). For any $x \in \mathbb{R}^n$, the closest point to $x$ in the span of the columns of $Q$ is
$$\hat x = QQ^Tx, \qquad \text{with coordinates } c = Q^Tx ,$$
and the error $x - \hat x$ is orthogonal to that span, so $\|x\|^2 = \|\hat x\|^2 + \|x - \hat x\|^2$.

<details>
<summary><b>Proof</b></summary>

Points of the span have the form $Qc$. Let $e = x - QQ^Tx$; then $Q^Te = Q^Tx - Q^TQQ^Tx = 0$. Write $x - Qc = e + Q(Q^Tx - c)$. The two pieces are orthogonal, and $\|Qz\| = \|z\|$, so
$$\|x - Qc\|^2 = \|e\|^2 + \|Q^Tx - c\|^2 ,$$
which is smallest exactly when $c = Q^Tx$. Setting $c = 0$ gives $\|x\|^2 = \|x - \hat x\|^2 + \|\hat x\|^2$. $\blacksquare$

</details>

This is how we compress **new** data with an existing basis $U_r$: store $c = U_r^Tx$ and rebuild $U_rc$. For PCA, subtract the mean first: $\hat x = \bar x + U_rU_r^T(x - \bar x)$.

*Careful:* $U_r^TU_r = I_r$, but $U_rU_r^T \neq I$ (unless $r = n$). It's a projection, and it throws information away.

**Theorem 3 (Eckart–Young).** Let $X_r = U_r\Sigma_rV_r^T$. For every matrix $B$ of rank at most $r$,
$$\|X - B\|_F \;\geq\; \|X - X_r\|_F \;=\; \sqrt{\sigma_{r+1}^2 + \sigma_{r+2}^2 + \cdots} .$$
Equivalently, the span of $u_1, \dots, u_r$ is the $r$-dimensional subspace closest to the columns of $X$. Since $\|X\|_F^2 = \sum_k \sigma_k^2$,
$$\frac{\|X - X_r\|_F^2}{\|X\|_F^2} = 1 - \underbrace{\frac{\sigma_1^2 + \cdots + \sigma_r^2}{\sum_k \sigma_k^2}}_{\text{fraction of the energy captured}} .$$
(Some texts plot the cumulative sum of $\sigma_k$ instead of $\sigma_k^2$. We use $\sigma_k^2$, which is what the theorem is about.)

<details>
<summary><b>Proof</b></summary>

**The error of $X_r$.** Orthogonal matrices preserve lengths, hence the Frobenius norm, so $\|X\|_F^2 = \|\Sigma\|_F^2 = \sum_k \sigma_k^2$. Likewise $X - X_r = \sum_{k>r} \sigma_k u_kv_k^T$ gives $\|X - X_r\|_F^2 = \sum_{k > r}\sigma_k^2$.

**Reduce to a subspace.** The columns of $B$ lie in some $r$-dimensional subspace; let $Q$ ($n \times r$) be an orthonormal basis for it. By Theorem 2, column by column,
$$\|X - B\|_F^2 \;\geq\; \|X - QQ^TX\|_F^2 \;=\; \|X\|_F^2 - \|Q^TX\|_F^2 .$$

**Bound what a subspace can capture.** With the full $U$ and $XX^T = \sum_{k=1}^n \sigma_k^2u_ku_k^T$,
$$\|Q^TX\|_F^2 = \operatorname{trace}(Q^TXX^TQ) = \sum_{k=1}^n \sigma_k^2 w_k, \qquad w_k = \|Q^Tu_k\|^2 .$$
Here $0 \leq w_k \leq 1$ and $\sum_k w_k = \|Q^TU\|_F^2 = \|Q\|_F^2 = r$. Such a weighted sum is largest when all the weight sits on the $r$ largest $\sigma_k^2$, so $\|Q^TX\|_F^2 \leq \sigma_1^2 + \cdots + \sigma_r^2$.

**Combine.** $\|X - B\|_F^2 \geq \sum_k\sigma_k^2 - \sum_{k \leq r}\sigma_k^2 = \|X - X_r\|_F^2$. Taking $Q = U_r$ gives equality, which proves the subspace version too. $\blacksquare$

</details>

## Toolbox

Everything we use in Part 5 (and all you need for your own data). `X` holds one data point per column.

| Math | NumPy |
|---|---|
| economy SVD $X = U\Sigma V^T$ | `U, S, Vt = np.linalg.svd(X, full_matrices=False)` |
| $\sigma_k$, $u_k$, $v_k$ | `S[k-1]`, `U[:, k-1]`, `Vt[k-1, :]` (Python counts from 0) |
| $U_r$, the first $r$ columns of $U$ | `U[:, :r]` |
| truncated SVD $X_r = U_r\Sigma_rV_r^T$ | `U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :]` |
| mean data point $\bar x$ | `x_bar = X.mean(axis=1)` |
| centered data $B$ (for PCA) | `B = X - x_bar[:, np.newaxis]` |
| coordinates $c = U_r^Tx$ | `c = U[:, :r].T @ x` |
| projection $U_rU_r^Tx$ | `U[:, :r] @ (U[:, :r].T @ x)` |
| energy fraction for every $r$ | `energy = np.cumsum(S**2) / np.sum(S**2)` (so `energy[r-1]` is for $r$ terms) |
| Frobenius norm $\lVert A \rVert_F$ | `np.linalg.norm(A)` |

**Watch out:**
- **Parentheses in projections.** `U[:, :r] @ (U[:, :r].T @ x)` is cheap. `(U[:, :r] @ U[:, :r].T) @ x` first builds an $n \times n$ matrix, which for a face photo has a billion entries (8 GB)!
- **Keep vectors 1D.** Mixing shapes `(n,)` and `(n, 1)` silently broadcasts to an $n \times n$ matrix. If a result has a strange size, print its `.shape`.

# Part 5: Applications

## 5.1 Compressing Mordecai

> **What's $X$?** The $2000 \times 1500$ dog. Its 1500 columns are data points in $\mathbb{R}^{2000}$, and compressing the dog means finding a low-dimensional subspace they all lie close to. We use the plain SVD here (no mean subtraction).

In [ ]:
X = load_gray(DATA_DIR + "/dog.jpg")                   # the steps from Part 1, packaged in a function
n, m = X.shape
U, S, Vt = np.linalg.svd(X, full_matrices=False)       # takes a second or two
print("U:", U.shape, "  S:", S.shape, "  Vt:", Vt.shape)

Storing the rank-$r$ approximation takes $r(n + m + 1)$ numbers ($U_r$, the $r$ singular values, and $V_r$) instead of the $nm$ pixels:

In [ ]:
ranks = [5, 20, 100]
approximations = [U[:, :r] @ np.diag(S[:r]) @ Vt[:r, :] for r in ranks]     # truncated SVDs

for r in ranks:
    storage = r * (n + m + 1)                  # numbers in U_r, the r singular values, and V_r
    print(f"r = {r:3d}: {storage:>7,} numbers instead of {n * m:,}  ->  {n * m / storage:5.1f} times smaller")

show_images([X] + approximations,
            ["original"] + [f"r = {r}  ({n * m / (r * (n + m + 1)):.0f}x smaller)" for r in ranks], size=4)

In [ ]:
energy = np.cumsum(S**2) / np.sum(S**2)        # energy[r - 1] = fraction of the energy in the first r terms
r99 = np.sum(energy < 0.99) + 1                # the smallest r that captures 99%

ks = np.arange(1, len(S) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].semilogy(ks, S, color=BLUE)
axes[0].set_xlabel("$k$"); axes[0].set_title(r"singular values $\sigma_k$ (log scale)")
axes[1].plot(ks, energy, color=BLUE)
axes[1].axvline(r99, color=GRAY, ls="--", label=f"99% of the energy: r = {r99}")
axes[1].set_xlabel("$r$"); axes[1].set_title("fraction of the energy captured by the first $r$ terms")
axes[1].legend()
plt.show()

Only a handful of singular values hold 99% of the energy, yet the rank-20 picture still looks blurry. Most of the energy (the sum of squared pixel values) sits in big bright areas like the snow, not in thin details like whiskers.

## 5.2 Eigenfaces

The Extended Yale Face Database B has 2,410 photos of 38 people, each under about 64 different lighting conditions. Every photo is $192 \times 168 = 32{,}256$ pixels.

> **What's $X$?** Each column is a **whole photo**, unrolled into a vector in $\mathbb{R}^{32{,}256}$ (the vector view from Part 1). We train on the first 36 people (2,282 photos) and keep the last two for testing. Each $u_k$ is itself an image: an **eigenface**. We subtract the average face first, so this is **PCA**.

In [ ]:
data = scipy.io.loadmat(DATA_DIR + "/allFaces.mat")   # a MATLAB file, loaded as a dictionary of arrays

faces = data["faces"]                          # 32256 x 2410, one photo per column (uint8 = whole numbers 0-255)
n = int(data["n"][0, 0])                       # image height (192). Stored as uint8, so convert to int:
m = int(data["m"][0, 0])                       # image width (168).  in uint8, 192 * 168 wraps around to 0!
nfaces = data["nfaces"].ravel().astype(int)    # number of photos of each of the 38 people
start = np.concatenate([[0], np.cumsum(nfaces)])    # person p's photos are columns start[p-1] to start[p]


def to_image(v):
    """Turn a vector of length 32256 back into a 192 x 168 picture.
    order="F": the photos were unrolled column by column (MATLAB's convention), not row by row."""
    return v.reshape(n, m, order="F")


def photos_of(p):
    """All photos of person p (p = 1, ..., 38), one per column."""
    return faces[:, start[p - 1]:start[p]]


show_images([to_image(photos_of(p)[:, 0]) for p in range(1, 37)], ncols=12, size=1.5)    # one photo per person

In [ ]:
N_TRAIN_PEOPLE = 36                                  # on a slow or low-memory laptop, try 20
train = faces[:, :start[N_TRAIN_PEOPLE]]             # training photos (uint8)

avg_face = train.mean(axis=1)                        # the average face, shape (32256,)
B = train - avg_face[:, np.newaxis]                  # centered data (decimals, since avg_face is decimal)
U, S, Vt = np.linalg.svd(B, full_matrices=False)     # the heaviest step: about 10 s on a fast laptop

show_images([to_image(avg_face)] + [to_image(U[:, k]) for k in range(7)],
            ["average face"] + [f"eigenface {k + 1}" for k in range(7)], size=3)

The first eigenfaces mostly capture **lighting** (from the left or right, above or below), since that's what varies most across the photos. Later ones capture features that tell people apart.

**A face the SVD has never seen.** Person 37 wasn't used for training. Rebuild one of their photos from $r$ eigenfaces with $\hat x = \bar x + U_rU_r^T(x - \bar x)$:

In [ ]:
x_test = photos_of(37)[:, 0].astype(float)
ranks = [r for r in [25, 50, 100, 200, 400, 800, 1600] if r <= U.shape[1]]
recon = [avg_face + U[:, :r] @ (U[:, :r].T @ (x_test - avg_face)) for r in ranks]
show_images([to_image(x_test)] + [to_image(x) for x in recon],
            ["original"] + [f"r = {r}  ({n * m / r:,.0f}x)" for r in ranks], size=2.5)

**A face is a point in $\mathbb{R}^r$.** The average face and the first $r$ eigenfaces are stored once and shared by everyone. After that, each face is described by just its $r$ coordinates $c = U_r^T(x - \bar x)$: $r$ numbers instead of $32{,}256$ pixels (the ratios in the titles above). For the whole training collection, storage drops from $32{,}256$ pixels per photo to $32{,}256\,(r + 1)$ numbers for the shared images, plus $r$ coordinates per photo:

In [ ]:
n_pixels = n * m                               # 32,256 pixels per photo
n_photos = B.shape[1]                          # number of training photos
for r in ranks:
    one_face = n_pixels / r                                                    # r coordinates instead of all the pixels
    collection = n_pixels * n_photos / (n_pixels * (r + 1) + r * n_photos)     # shared images + r coordinates per photo
    print(f"r = {r:4d}: one face is {r:4d} numbers ({one_face:5,.0f}x smaller).  "
          f"All {n_photos:,} photos: {collection:4.1f}x smaller")

**Telling people apart.** The coordinates of all photos of persons 2 and 7 along eigenfaces 5 and 6 form two separate clusters. That's the starting point for face recognition, which comes up later in the book. (Try other pairs of eigenfaces: eigenfaces 1 and 2 do much worse. Why?)

In [ ]:
modes = [4, 5]                                                   # eigenfaces 5 and 6 (Python counts from 0)
for p, color, marker in [(2, BLUE, "D"), (7, ORANGE, "^")]:
    coords = U[:, modes].T @ (photos_of(p) - avg_face[:, np.newaxis])    # coordinates of each photo
    plt.scatter(coords[0], coords[1], color=color, marker=marker, label=f"person {p}")
plt.xlabel(f"coordinate along eigenface {modes[0] + 1}")
plt.ylabel(f"coordinate along eigenface {modes[1] + 1}")
plt.legend(); plt.show()

**Is Mordecai a face?** Crop the dog to his head, shrink it to $192 \times 168$, and rebuild it from eigenfaces:

In [ ]:
from PIL import Image                     # the image library that matplotlib uses behind the scenes

dog = load_gray(DATA_DIR + "/dog.jpg")[60:1204, 0:1001]                     # crop to the head
dog_small = np.asarray(Image.fromarray(dog.astype(np.uint8)).resize((m, n)), dtype=float)   # (width, height)
x_dog = dog_small.reshape(-1, order="F")                                    # unroll like the faces

recon = [avg_face + U[:, :r] @ (U[:, :r].T @ (x_dog - avg_face)) for r in ranks]
show_images([dog_small] + [to_image(x) for x in recon], ["original"] + [f"r = {r}" for r in ranks], size=2.5)

With a few eigenfaces, the dog becomes a ghostly face: that's all the first eigenfaces can build. With many, the basis gets rich enough to approximate almost any picture.

## 5.3 Hot rods: data from physics

Data often comes from **functions**: a temperature along a rod, the airflow around a wing. A computer stores a function by its values at $n$ points, which is a vector in $\mathbb{R}^n$, so everything above applies.

**The physics.** A metal rod of length 1 has both ends held in ice water at 0 °C. A heater centered at position $c$ (with width $w$ and power $P$) warms it. Once the temperature $u(x)$ stops changing, it satisfies the steady **heat equation**
$$-u''(x) = f(x), \qquad u(0) = u(1) = 0,$$
where $f$ is the heat input. We simulate 300 rods with randomly placed heaters, and store each rod's temperature at 201 points as one column of a matrix `T`.

<details>
<summary><b>How the data is made</b> (optional)</summary>

Put 201 equally spaced points on the rod, with spacing $h = 1/200$, and replace $u''$ by the **finite difference** $u''(x_j) \approx (u_{j-1} - 2u_j + u_{j+1})/h^2$. At the 199 interior points, the heat equation becomes a linear system $Lu = f$ with the tridiagonal matrix
$$L = \frac{1}{h^2}\begin{bmatrix} 2 & -1 & & \\ -1 & 2 & \ddots & \\ & \ddots & \ddots & -1 \\ & & -1 & 2 \end{bmatrix}.$$
Each heater is a bump $f(x) = \dfrac{P}{w\sqrt{2\pi}}\, e^{-(x-c)^2/(2w^2)}$ with total power $P$, and random $c$, $w$ and $P$.

</details>

In [ ]:
# ---- Heat-rod simulator (see "How the data is made" above) ----
x = np.linspace(0, 1, 201)                     # 201 points along the rod
h = x[1] - x[0]
n_inside = len(x) - 2                          # interior points (the ends are fixed at 0 degrees)
L = (2 * np.eye(n_inside) - np.eye(n_inside, k=1) - np.eye(n_inside, k=-1)) / h**2    # -d^2/dx^2


def heater(c, w, P):
    """Heat input along the rod, one column per heater: bumps at positions c, with widths w and total powers P."""
    c, w, P = np.atleast_1d(c, w, P)
    return P / (w * np.sqrt(2 * np.pi)) * np.exp(-(x[:, np.newaxis] - c)**2 / (2 * w**2))


def rod_temperature(f):
    """Steady temperature of the rod (0 at both ends) for the heat input f (one rod per column)."""
    u = np.zeros(f.shape)
    u[1:-1] = np.linalg.solve(L, f[1:-1])
    return u


rng = np.random.default_rng(8)
n_rods = 300
positions = rng.uniform(0.1, 0.9, n_rods)
F = heater(positions, rng.uniform(0.01, 0.1, n_rods), rng.uniform(50, 150, n_rods))   # heat inputs
T = rod_temperature(F)                                                                # temperatures

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for j, color in zip(range(3), [BLUE, ORANGE, GREEN]):
    axes[0].plot(x, F[:, j], color=color, label=f"rod {j + 1}")
    axes[1].plot(x, T[:, j], color=color, label=f"rod {j + 1}")
axes[0].set_title("heat input $f(x)$"); axes[0].set_xlabel("position $x$"); axes[0].legend()
axes[1].set_title("temperature $u(x)$ in °C"); axes[1].set_xlabel("position $x$"); axes[1].legend()
order = np.argsort(positions)                  # sort the rods by heater position, just for this picture
im = axes[2].imshow(T[:, order], aspect="auto", cmap="inferno", extent=[0.5, n_rods + 0.5, 1, 0])
axes[2].set_title("the data matrix $T$ (201 x 300)")
axes[2].set_xlabel("rod (sorted by heater position)"); axes[2].set_ylabel("position $x$")
plt.colorbar(im, ax=axes[2], label="°C")
plt.show()

> **What's $X$?** The matrix `T`: each of its 300 columns is a temperature profile in $\mathbb{R}^{201}$. We use the plain SVD; in engineering this is called the *proper orthogonal decomposition* (POD).

In [ ]:
U, S, Vt = np.linalg.svd(T, full_matrices=False)
energy = np.cumsum(S**2) / np.sum(S**2)
print("modes needed for 99.99% of the energy:", np.sum(energy < 0.9999) + 1, "out of", len(S))

S_F = np.linalg.svd(F, compute_uv=False)                     # singular values of the heat inputs, for comparison
ks = np.arange(1, 81)
plt.semilogy(ks, S[:80] / S[0], color=BLUE, label="temperatures $T$")
plt.semilogy(ks, S_F[:80] / S_F[0], color=ORANGE, label="heat inputs $F$")
plt.xlabel("$k$"); plt.title(r"singular values, scaled: $\sigma_k / \sigma_1$"); plt.legend()
plt.show()

The temperatures are far more compressible than the heat inputs. Heat spreads out: solving the heat equation smooths away sharp features, and smooth curves are built from a few smooth patterns. The modes themselves look like sine waves $\sin(k\pi x)$, the natural patterns of $-\frac{d^2}{dx^2}$ with zero ends. The SVD found them from data alone.

In [ ]:
for k, color in zip(range(4), [BLUE, ORANGE, GREEN, YELLOW]):
    plt.plot(x, U[:, k], color=color, label=f"$u_{k + 1}$")
plt.xlabel("position $x$"); plt.legend()
plt.show()

**A new rod**, with a heater at $c = 0.37$, $w = 0.03$, $P = 120$, which is not in the data. How well do $r$ numbers describe it?

In [ ]:
u_new = rod_temperature(heater(0.37, 0.03, 120))[:, 0]        # temperature profile of the new rod

fig, axes = plt.subplots(1, 4, figsize=(18, 3.8), sharey=True)
for ax, r in zip(axes, [1, 2, 4, 8]):
    coeffs = U[:, :r].T @ u_new                               # its r coordinates
    ax.plot(x, u_new, color=GRAY, lw=5, label="true temperature")
    ax.plot(x, U[:, :r] @ coeffs, color=BLUE, lw=1.5, label=f"from {r} numbers")
    ax.set_xlabel("$x$"); ax.legend(loc="upper right", fontsize=8)
plt.show()

A handful of numbers describes the temperature of a rod we'd never seen. Later in the book, this idea leads to **reduced-order models**, which solve the physics directly in that small space.

# Summary

- **Data are vectors:** a photo, a face or a temperature profile is a point in $\mathbb{R}^n$.
- **Real data lies close to a low-dimensional subspace.** The SVD finds the best one (keep $u_1, \dots, u_r$), and by Eckart–Young the error is the leftover $\sigma_{r+1}^2 + \sigma_{r+2}^2 + \cdots$.
- **PCA = subtract the mean, then take the SVD.** Use it when you care about variation around the average.
- **New data** is compressed with $c = U_r^Tx$ and rebuilt with $U_rc$.
- The **toolbox** at the end of Part 4 has every command.

**Try it yourself:**
- Compress your own photo with `load_gray("your_photo.jpg")`. Do photos with fine texture compress worse than ones with smooth regions?
- Compress the color photo: one SVD for each of the red, green and blue channels.
- Heat a rod with two heaters at once (`heater(...) + heater(...)`). Does the 4-mode approximation still work? Why? (Hint: the heat equation is linear.)

The book's website, [databookuw.com](https://databookuw.com/), has video lectures and Python code.